# gARB Pipeline Validation
Run each cell in order. Confirm every step works before the hackathon.

**If any cell fails, stop and fix it. Do not move on.**

## Step 1: Fetch DEM from USGS 3DEP

In [ ]:
import py3dep
import matplotlib.pyplot as plt
import numpy as np

# Ellerbe Creek watershed bounding box
bbox = (-79.05, 35.90, -78.75, 36.05)

print('Fetching DEM...')
dem = py3dep.get_map('DEM', bbox, resolution=10)
print(f'DEM shape: {dem.shape}')
print(f'Elevation range: {float(dem.min()):.1f}m to {float(dem.max()):.1f}m')

plt.figure(figsize=(12, 8))
dem.plot(cmap='terrain')
plt.title('Ellerbe Creek Watershed DEM')
plt.show()
print('✓ Step 1 passed')

## Step 2: Hydrological Processing (pysheds)

In [ ]:
from pysheds.grid import Grid

# Extract arrays from xarray
values = dem.values.squeeze()
transform = dem.rio.transform()
crs_str = str(dem.rio.crs)

grid = Grid()
grid.add_gridded_data(values, data_name='dem', affine=transform, crs=crs_str,
                      nodata=dem.rio.nodata or -9999)

print('Filling pits...')
pit_filled = grid.fill_pits('dem')
grid.add_gridded_data(pit_filled, data_name='pit_filled', affine=transform, crs=crs_str)

print('Filling depressions...')
flooded = grid.fill_depressions('pit_filled')
grid.add_gridded_data(flooded, data_name='flooded', affine=transform, crs=crs_str)

print('Resolving flats...')
inflated = grid.resolve_flats('flooded')
grid.add_gridded_data(inflated, data_name='inflated', affine=transform, crs=crs_str)

print('Computing flow direction...')
fdir = grid.flowdir('inflated')
grid.add_gridded_data(fdir, data_name='fdir', affine=transform, crs=crs_str)

print('Computing flow accumulation...')
acc = grid.accumulation('fdir')
grid.add_gridded_data(acc, data_name='acc', affine=transform, crs=crs_str)

print(f'Max accumulation: {np.nanmax(acc):.0f} cells')

plt.figure(figsize=(12, 8))
plt.imshow(np.log10(acc + 1), cmap='Blues')
plt.colorbar(label='log10(accumulation)')
plt.title('Flow Accumulation (log scale)')
plt.show()
print('✓ Step 2 passed')

## Step 3: Extract Stream Network

In [ ]:
import json

print('Extracting stream network (threshold=500)...')
streams = grid.extract_river_network('fdir', 'acc', threshold=500)

n_features = len(streams.get('features', []))
print(f'Extracted {n_features} stream segments')

# Save as GeoJSON
with open('../mock_data/streams.geojson', 'w') as f:
    json.dump(streams, f)
print('Saved to mock_data/streams.geojson')
print('✓ Step 3 passed')
print()
print('>>> Open mock_data/streams.geojson in geojson.io to visually confirm')
print('>>> The streams should follow the Ellerbe Creek path through Durham')

## Step 4: Generate & Score Candidates

In [ ]:
import geopandas as gpd
from shapely.geometry import shape

# Convert streams to GeoDataFrame
features = streams.get('features', [])
geometries = [shape(f['geometry']) for f in features]
stream_gdf = gpd.GeoDataFrame({'geometry': geometries}, crs='EPSG:4326')

print(f'Stream GeoDataFrame: {len(stream_gdf)} segments')
print(f'Total stream length: {stream_gdf.to_crs("EPSG:32617").geometry.length.sum()/1000:.1f} km')

# Generate candidates
from core.pipeline import generate_candidates, compute_strahler_order

stream_gdf = compute_strahler_order(stream_gdf)
candidates = generate_candidates(stream_gdf, spacing_m=200)

print(f'Generated {len(candidates)} candidate sites')

# Save
candidates.to_crs('EPSG:4326').to_file('../mock_data/candidates_raw.geojson', driver='GeoJSON')
print('Saved raw candidates to mock_data/candidates_raw.geojson')
print('✓ Step 4 passed')

## Step 5: Test API Connectivity (EPA, USGS, Census)

In [ ]:
import requests

apis = {
    'USGS NWIS': 'https://waterservices.usgs.gov/nwis/iv/?format=json&sites=02086849&parameterCd=00060&period=P1D',
    'EPA ECHO': 'https://echo.epa.gov/api/facility-search/facilities?output=JSON&p_st=NC&p_permit_type=NPD&qcolumns=1,3',
    'Census ACS': 'https://api.census.gov/data/2022/acs/acs5?get=B01003_001E&for=state:37',
}

for name, url in apis.items():
    try:
        r = requests.get(url, timeout=15)
        print(f'{name}: {r.status_code} ({len(r.content)} bytes) ✓')
    except Exception as e:
        print(f'{name}: FAILED — {e} ✗')

print()
print('If all APIs return 200, you\'re ready for live scoring.')
print('If any fail, mock data still works for the demo.')

## Done

If all 5 steps passed:
- Your DEM pipeline works
- Stream extraction produces real Ellerbe Creek geometry
- Candidate generation creates valid GeoJSON
- External APIs are reachable

Run `python scripts/generate_mock.py` to create the safety-net mock data.

Then start the API: `uvicorn api.main:app --reload --port 8000`

Then open `dashboard/index.html`.